In [88]:
import numpy as np
import pandas as pd
from pathlib import Path
import warnings; warnings.filterwarnings("ignore")

from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve, brier_score_loss
from sklearn.calibration import calibration_curve
from sklearn.isotonic import IsotonicRegression
import matplotlib.pyplot as plt


In [89]:
DATA_DIR = Path("./Data")   # <-- *pointed* at the folder with the three CSVs

def load(split):
    df = pd.read_csv(DATA_DIR / f"earthquakeq_{split}.csv", parse_dates=["time"])
    return df

train_raw = load("train")

s = seismic(train_raw)

print(s.columns.tolist())


test_raw  = load("test")
judge_raw = load("judge")

for name, df in [("train", train_raw), ("test", test_raw), ("judge", judge_raw)]:
    print(f"{name:6s}: {len(df):>7,} rows | sources: {dict(df.source.value_counts())}")

['time', 'latitude', 'longitude', 'depth_km', 'magnitude', 'coast delta', 'fault delta']
train :  11,597 rows | sources: {'seismic': np.int64(11597)}
test  :   2,027 rows | sources: {'seismic': np.int64(2027)}
judge :     740 rows | sources: {'seismic': np.int64(740)}


In [90]:
def seismic(df):
    s = df[
        [
            "time",
            "latitude",
            "longitude",
            "depth_km",
            "magnitude",
            "coast delta",
            "fault delta",
        ]
    ].copy()

    return s.sort_values("time").reset_index(drop=True)


## 7 · Your turn

You now have a complete, honest, working forecast. It is **plain on purpose.** Some directions teams
often explore (no promises — you decide what's worth it):

- **Richer seismicity features** — b-values, event-rate acceleration, moment release, multi-radius counts.
- **The other four data streams** — geomagnetic, solar, tidal, GPS. Do they add anything? Prove it, don't assume it.
- **A physics baseline** — the ETAS aftershock model is a classic; feeding its forecast in as a feature is a strong move.
- **Better calibration and model selection** — how you split time and calibrate can matter more than the model.
- **Quantum approaches** — encodings, kernels, variational classifiers, if your track calls for it.

**Two things worth remembering:**
1. Report your gains *with* calibration, and always check your held-out (judge) number against your
   own test number — if judge is far worse, you probably leaked.
2. If your fancy pipeline doesn't clearly beat this simple one, that's a real and interesting result.
   Say so, honestly, with the numbers to back it up.

Good luck. Build something you can defend.

QBits:

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import torch
import pennylane as qml

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    log_loss,
    brier_score_loss,
    confusion_matrix,
)


from pennylane.qnn import TorchLayer








def inspect_labels(name, y):
    values, counts = np.unique(y, return_counts=True)

    print(f"\n{name}")
    print("shape:", y.shape)
    print("unique values:", values)
    print("counts:", dict(zip(values, counts)))
    print("positive rate:", float(np.mean(y)))









# ============================================================
# Configuration
# ============================================================

SEED = 69
N_QUBITS = 6
N_LAYERS = 3

MAGNITUDE_THRESHOLD = 2.0
RADIUS_KM = 100.0
HORIZON_DAYS = 30

BATCH_SIZE = 32
EPOCHS = 4
LEARNING_RATE = 0.025
CLASSIFICATION_THRESHOLD = 0.5

# Use a simulator, not IBM hardware
SHOTS = 128

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


# ============================================================
# Basic validation and geographic utilities
# ============================================================

def validate_columns(df, name):
    required = {"time", "latitude", "longitude", "magnitude"}
    missing = required - set(df.columns)

    if missing:
        raise ValueError(
            f"{name} is missing required columns: {sorted(missing)}. "
            f"Available columns are: {list(df.columns)}"
        )


def haversine_km(lat1, lon1, lat2, lon2):
    """
    Calculate great-circle distance in kilometers.
    lat1/lon1 may be scalars; lat2/lon2 may be NumPy arrays.
    """
    earth_radius_km = 6371.0088

    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2.0) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    )

    return 2.0 * earth_radius_km * np.arcsin(np.sqrt(a))


# ============================================================
# Data preparation
# ============================================================

def clean_catalog(df, name):
    df = df.copy()
    validate_columns(df, name)

    # Convert everything to UTC, then remove timezone information.
    # This makes all timestamps consistently timezone-naive.
    df["time"] = (
        pd.to_datetime(df["time"], errors="coerce", utc=True)
        .dt.tz_localize(None)
    )

    numeric_columns = [
        "latitude",
        "longitude",
        "magnitude",
        "depth_km",
        "coast delta",
        "fault delta",
    ]

    for column in numeric_columns:
        df[column] = pd.to_numeric(
            df[column],
            errors="coerce",
        )

    df = df.dropna(
        subset=[
            "time",
            "latitude",
            "longitude",
            "magnitude",
        ]
    )

    return df.sort_values("time").reset_index(drop=True)



def make_future_event_labels(
    reference_df,
    earthquake_df,
    radius_km=100.0,
    horizon_days=100,
    magnitude_threshold=2.0,
):
    """
    Label each reference row as 1 if an earthquake with magnitude greater
    than magnitude_threshold occurs within radius_km during the future
    horizon_days.
    """

    reference_df = reference_df.copy()
    earthquake_df = earthquake_df.copy()

    # Normalize timestamps again inside the function for safety.
    reference_df["time"] = (
        pd.to_datetime(
            reference_df["time"],
            errors="coerce",
            utc=True,
        )
        .dt.tz_localize(None)
    )

    earthquake_df["time"] = (
        pd.to_datetime(
            earthquake_df["time"],
            errors="coerce",
            utc=True,
        )
        .dt.tz_localize(None)
    )

    reference_df = (
        reference_df
        .dropna(subset=["time", "latitude", "longitude"])
        .reset_index(drop=True)
    )

    earthquake_df = (
        earthquake_df
        .dropna(
            subset=[
                "time",
                "latitude",
                "longitude",
                "magnitude",
            ]
        )
        .sort_values("time")
        .reset_index(drop=True)
    )

    event_times = earthquake_df["time"]
    event_latitudes = earthquake_df["latitude"].to_numpy()
    event_longitudes = earthquake_df["longitude"].to_numpy()
    event_magnitudes = earthquake_df["magnitude"].to_numpy()

    labels = np.zeros(
        len(reference_df),
        dtype=np.int64,
    )

    for i, reference_row in reference_df.iterrows():
        start_time = reference_row["time"]
        end_time = (
            start_time
            + pd.Timedelta(days=horizon_days)
        )

        # Use pandas Series comparisons. All timestamps are now
        # timezone-naive, so these comparisons are compatible.
        candidate_mask = (
            (event_times > start_time)
            & (event_times <= end_time)
            & (event_magnitudes > magnitude_threshold)
        )

        candidate_indices = np.flatnonzero(
            candidate_mask.to_numpy()
        )

        if len(candidate_indices) == 0:
            continue

        distances = haversine_km(
            reference_row["latitude"],
            reference_row["longitude"],
            event_latitudes[candidate_indices],
            event_longitudes[candidate_indices],
        )

        labels[i] = int(
            np.any(distances <= radius_km)
        )

    return labels



def add_time_features(df):
    """
    Add cyclical and numeric time features.
    """
    df = df.copy()

    time = df["time"]

    df["time_unix_days"] = (
        time.astype("int64") / 10**9 / 86400.0
    )

    df["hour"] = time.dt.hour
    df["day_of_week"] = time.dt.dayofweek
    df["day_of_year"] = time.dt.dayofyear
    df["month"] = time.dt.month

    # Cyclical encodings
    df["hour_sin"] = np.sin(2.0 * np.pi * df["hour"] / 24.0)
    df["hour_cos"] = np.cos(2.0 * np.pi * df["hour"] / 24.0)

    df["day_of_week_sin"] = np.sin(
        2.0 * np.pi * df["day_of_week"] / 7.0
    )
    df["day_of_week_cos"] = np.cos(
        2.0 * np.pi * df["day_of_week"] / 7.0
    )

    df["day_of_year_sin"] = np.sin(
        2.0 * np.pi * df["day_of_year"] / 365.25
    )
    df["day_of_year_cos"] = np.cos(
        2.0 * np.pi * df["day_of_year"] / 365.25
    )

    df["month_sin"] = np.sin(
        2.0 * np.pi * df["month"] / 12.0
    )
    df["month_cos"] = np.cos(
        2.0 * np.pi * df["month"] / 12.0
    )

    return df


def make_feature_matrix(df):
    """
    Build features available at the reference time.

    Current magnitude is excluded to avoid making the target too easy
    or accidentally using a target-related value as an input.
    """
    df = add_time_features(df)

    feature_columns = [
        "latitude",
        "longitude",
        "depth_km",
        "coast delta",
        "fault delta",
        "time_unix_days",
        "hour_sin",
        "hour_cos",
        "day_of_week_sin",
        "day_of_week_cos",
        "day_of_year_sin",
        "day_of_year_cos",
        "month_sin",
        "month_cos",
    ]


    missing = set(feature_columns) - set(df.columns)

    if missing:
        raise ValueError(f"Missing feature columns: {sorted(missing)}")

    return df[feature_columns].to_numpy(dtype=np.float32)


# ============================================================
# Load and prepare the catalogs
# ============================================================

# These functions/objects must already exist in your environment:
#
# train_raw
# test_raw
# judge_raw
# seismic(...)
#

cat_train = clean_catalog(seismic(train_raw), "training catalog")
cat_test = clean_catalog(seismic(test_raw), "test catalog")
cat_judge = clean_catalog(seismic(judge_raw), "judge catalog")


# ============================================================
# Create binary labels
# ============================================================

y_train = make_future_event_labels(
    reference_df=cat_train,
    earthquake_df=cat_train,
    radius_km=RADIUS_KM,
    horizon_days=HORIZON_DAYS,
    magnitude_threshold=MAGNITUDE_THRESHOLD,
)

y_test = make_future_event_labels(
    reference_df=cat_test,
    earthquake_df=cat_test,
    radius_km=RADIUS_KM,
    horizon_days=HORIZON_DAYS,
    magnitude_threshold=MAGNITUDE_THRESHOLD,
)

y_judge = make_future_event_labels(
    reference_df=cat_judge,
    earthquake_df=cat_judge,
    radius_km=RADIUS_KM,
    horizon_days=HORIZON_DAYS,
    magnitude_threshold=MAGNITUDE_THRESHOLD,
)

# ============================================================
# Create feature matrices
# ============================================================

x_train = make_feature_matrix(cat_train)
x_test = make_feature_matrix(cat_test)
x_judge = make_feature_matrix(cat_judge)


# ============================================================
# Chronological split
# ============================================================

# Sort each split chronologically before preprocessing.
# Do not randomly shuffle the full time series before the split.
train_order = np.argsort(cat_train["time"].to_numpy())
test_order = np.argsort(cat_test["time"].to_numpy())
judge_order = np.argsort(cat_judge["time"].to_numpy())

x_train = x_train[train_order]
y_train = y_train[train_order]

x_test = x_test[test_order]
y_test = y_test[test_order]

x_judge = x_judge[judge_order]
y_judge = y_judge[judge_order]

# ============================================================
# Scale and reduce features
# ============================================================

from sklearn.impute import SimpleImputer

imputer_x = SimpleImputer(strategy="median")

x_train_imputed = imputer_x.fit_transform(x_train)
x_test_imputed = imputer_x.transform(x_test)
x_judge_imputed = imputer_x.transform(x_judge)

scaler_x = StandardScaler()

x_train_scaled = scaler_x.fit_transform(x_train_imputed)
x_test_scaled = scaler_x.transform(x_test_imputed)
x_judge_scaled = scaler_x.transform(x_judge_imputed)


from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

baseline = LogisticRegression(max_iter=1000)
baseline.fit(x_train_scaled, y_train)

baseline_pred = baseline.predict(x_test_scaled)

# print(classification_report(y_test, baseline_pred, zero_division=0))


# PCA reduces the classical features to exactly N_QUBITS inputs.
pca = PCA(n_components=N_QUBITS)

x_train_reduced = pca.fit_transform(x_train_scaled)
x_test_reduced = pca.transform(x_test_scaled)
x_judge_reduced = pca.transform(x_judge_scaled)


# ============================================================
# Convert data to PyTorch tensors
# ============================================================

X_train_t = torch.tensor(
    x_train_reduced,
    dtype=torch.float32
)

X_test_t = torch.tensor(
    x_test_reduced,
    dtype=torch.float32
)

X_judge_t = torch.tensor(
    x_judge_reduced,
    dtype=torch.float32
)

Y_train_t = torch.tensor(
    y_train.reshape(-1, 1),
    dtype=torch.float32
)

Y_test_t = torch.tensor(
    y_test.reshape(-1, 1),
    dtype=torch.float32
)

Y_judge_t = torch.tensor(
    y_judge.reshape(-1, 1),
    dtype=torch.float32
)


# ============================================================
# Training data loader
# ============================================================

dataset_train = torch.utils.data.TensorDataset(
    X_train_t,
    Y_train_t,
)

loader_train = torch.utils.data.DataLoader(
    dataset_train,
    batch_size=BATCH_SIZE,
    shuffle=True,
)


# ============================================================
# PennyLane simulator quantum circuit
# ============================================================

# This is a local simulator.
# No IBM account, token, backend, or network connection is needed.
dev = qml.device(
    "default.qubit",
    wires=N_QUBITS,
    shots=SHOTS,
)


@qml.qnode(
    dev,
    interface="torch",
    diff_method="parameter-shift",
)
def circuit(inputs, weights):
    qml.AngleEmbedding(
        inputs,
        wires=range(N_QUBITS),
    )

    qml.BasicEntanglerLayers(
        weights,
        wires=range(N_QUBITS),
    )

    return [qml.expval(qml.PauliZ(i)) for i in range(N_QUBITS)]


# ============================================================
# Hybrid quantum-classical classifier
# ============================================================

weight_shapes = {
    "weights": (N_LAYERS, N_QUBITS)
}

q_layer = TorchLayer(
    circuit,
    weight_shapes,
)


class HybridClassifier(torch.nn.Module):
    def __init__(self, quantum_layer):
        super().__init__()

        self.quantum_layer = quantum_layer
        self.output_layer = torch.nn.Linear(N_QUBITS, 1)

    def forward(self, x):
        quantum_output = self.quantum_layer(x)

        # Output is a raw logit, not a probability
        return self.output_layer(quantum_output)


model = HybridClassifier(q_layer)

# BCEWithLogitsLoss applies the sigmoid internally.
# It is numerically preferable to applying sigmoid before the loss.
loss_fn = torch.nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
)


# ============================================================
# Train
# ============================================================

last_loss = 0

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0

    for xb, yb in loader_train:
        optimizer.zero_grad()

        logits = model(xb)
        loss = loss_fn(logits, yb)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(xb)

    average_loss = total_loss / len(dataset_train)
    if round(average_loss, 4) == last_loss:
        break

    last_loss = round(average_loss, 4)
    print(
        f"Epoch {epoch + 1:02d}/{EPOCHS} "
        f"- loss: {average_loss:.4f}"
    )



# ============================================================
# Additional evaluation metrics
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,   # PR-AUC
    log_loss,
    brier_score_loss,
    confusion_matrix,
)


def expected_calibration_error(
    y_true,
    probabilities,
    n_bins=10,
):
    """
    Expected Calibration Error (ECE).

    Probabilities are divided into equally spaced bins between
    0 and 1. ECE is the weighted average absolute difference
    between mean predicted probability and empirical accuracy
    in each non-empty bin.
    """

    y_true = np.asarray(y_true).reshape(-1)
    probabilities = np.asarray(probabilities).reshape(-1)

    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)

    ece = 0.0
    n = len(y_true)

    for i in range(n_bins):
        if i == n_bins - 1:
            mask = (
                (probabilities >= bin_edges[i])
                & (probabilities <= bin_edges[i + 1])
            )
        else:
            mask = (
                (probabilities >= bin_edges[i])
                & (probabilities < bin_edges[i + 1])
            )

        if not np.any(mask):
            continue

        bin_probabilities = probabilities[mask]
        bin_labels = y_true[mask]

        mean_probability = np.mean(bin_probabilities)
        empirical_accuracy = np.mean(bin_labels)

        ece += (
            np.sum(mask) / n
        ) * abs(
            mean_probability - empirical_accuracy
        )

    return float(ece)


def evaluate_model(
    model,
    X,
    y,
    threshold=0.5,
    baseline_positive_rate=None,
    ece_bins=10,
):
    model.eval()

    with torch.no_grad():
        logits = model(X)

        probabilities = (
            torch.sigmoid(logits)
            .reshape(-1)
            .cpu()
            .numpy()
        )

    predictions = (
        probabilities >= threshold
    ).astype(np.int64)

    y_true = y.reshape(-1).astype(np.int64)

    # --------------------------------------------------------
    # Classification metrics
    # --------------------------------------------------------

    results = {
        "accuracy": accuracy_score(
            y_true,
            predictions,
        ),

        "precision": precision_score(
            y_true,
            predictions,
            zero_division=0,
        ),

        "recall": recall_score(
            y_true,
            predictions,
            zero_division=0,
        ),

        "f1": f1_score(
            y_true,
            predictions,
            zero_division=0,
        ),

        "confusion_matrix": confusion_matrix(
            y_true,
            predictions,
        ),

        "probabilities": probabilities,

        "predictions": predictions,
    }

    # --------------------------------------------------------
    # ROC-AUC
    # --------------------------------------------------------

    if len(np.unique(y_true)) == 2:
        results["auc"] = roc_auc_score(
            y_true,
            probabilities,
        )

        # Average precision is the standard sklearn
        # implementation commonly used for PR-AUC.
        results["pr_auc"] = average_precision_score(
            y_true,
            probabilities,
        )
    else:
        results["auc"] = float("nan")
        results["pr_auc"] = float("nan")

    # --------------------------------------------------------
    # Brier score
    # --------------------------------------------------------

    results["brier"] = brier_score_loss(
        y_true,
        probabilities,
    )

    # --------------------------------------------------------
    # Log loss and Information Gain
    # --------------------------------------------------------

    # Clip probabilities very slightly to avoid log(0).
    probabilities_clipped = np.clip(
        probabilities,
        1e-15,
        1.0 - 1e-15,
    )

    model_log_loss = log_loss(
        y_true,
        probabilities_clipped,
        labels=[0, 1],
    )

    # If no external baseline rate is supplied, use the
    # empirical positive rate of this evaluation set.
    if baseline_positive_rate is None:
        baseline_positive_rate = float(np.mean(y_true))

    baseline_positive_rate = np.clip(
        baseline_positive_rate,
        1e-15,
        1.0 - 1e-15,
    )

    baseline_probabilities = np.full(
        len(y_true),
        baseline_positive_rate,
        dtype=np.float64,
    )

    baseline_log_loss = log_loss(
        y_true,
        baseline_probabilities,
        labels=[0, 1],
    )

    # Convert nats to bits.
    results["ig_bits"] = (
        baseline_log_loss - model_log_loss
    ) / np.log(2.0)

    results["log_loss"] = model_log_loss

    # --------------------------------------------------------
    # Expected Calibration Error
    # --------------------------------------------------------

    results["ece"] = expected_calibration_error(
        y_true=y_true,
        probabilities=probabilities,
        n_bins=ece_bins,
    )

    return results


# ============================================================
# Evaluate on test and judge sets
# ============================================================

# Use the training-set event rate as the baseline for IG_bits.
train_positive_rate = float(
    np.mean(y_train)
)

print(
    f"\nTraining positive rate: "
    f"{train_positive_rate:.6f}"
)


test_results = evaluate_model(
    model=model,
    X=X_test_t,
    y=y_test,
    threshold=0.65,
    baseline_positive_rate=train_positive_rate,
    ece_bins=10,
)


judge_results = evaluate_model(
    model=model,
    X=X_judge_t,
    y=y_judge,
    threshold=CLASSIFICATION_THRESHOLD,
    baseline_positive_rate=train_positive_rate,
    ece_bins=10,
)


# ============================================================
# Test results
# ============================================================

print("\nTest results")

print(f"AUC:         {test_results['auc']:.4f}")
print(f"PR-AUC:      {test_results['pr_auc']:.4f}")
print(f"IG_bits:     {test_results['ig_bits']:.4f}")
print(f"Brier:       {test_results['brier']:.4f}")
print(f"ECE:         {test_results['ece']:.4f}")

print(f"Accuracy:    {test_results['accuracy']:.4f}")
print(f"Precision:   {test_results['precision']:.4f}")
print(f"Recall:      {test_results['recall']:.4f}")
print(f"F1 score:    {test_results['f1']:.4f}")

print("Confusion matrix:")
print(test_results["confusion_matrix"])


# ============================================================
# Judge results
# ============================================================

print("\nJudge results")

print(f"AUC:         {judge_results['auc']:.4f}")
print(f"PR-AUC:      {judge_results['pr_auc']:.4f}")
print(f"IG_bits:     {judge_results['ig_bits']:.4f}")
print(f"Brier:       {judge_results['brier']:.4f}")
print(f"ECE:         {judge_results['ece']:.4f}")

print(f"Accuracy:    {judge_results['accuracy']:.4f}")
print(f"Precision:   {judge_results['precision']:.4f}")
print(f"Recall:      {judge_results['recall']:.4f}")
print(f"F1 score:    {judge_results['f1']:.4f}")

print("Confusion matrix:")
print(judge_results["confusion_matrix"])



# ============================================================
# Example: inspect predictions
# ============================================================

judge_predictions = pd.DataFrame({
    "actual": y_judge,
    "probability_event_within_100km": judge_results["probabilities"],
    "predicted_class": judge_results["predictions"],
})

print("\nPrediction sample:")
print(judge_predictions.head(10))
